# dicom-extremities-preprocessor

**Summary**

In:
- 8 raw DICOMs in `sample_data/`, one special case each

Out:
- `output/dicoms/dp/`, `output/dicoms/oblique/`
- `output/csvs/pairs.csv` -- one row per case
- `output/provenance.json`

Steps:

```
1.  read headers, no pixels    header.scan_metadata
2.  derive categories          rules.categorize
3.  select bodypart / view     rules.select
3b. two hands on one film?     pixels.detect_bilateral
4.  standardize + write        pixels.write_processed
5.  pair the views             pipeline.build_pairs
```


Decisionrules in `config/rules.yaml`

In [ ]:
import os, sys, json, shutil, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt

HERE = Path.cwd() if Path.cwd().name == "demo" else Path.cwd() / "demo"
sys.path.insert(0, str(HERE.parent))          # package sits one level up

import dicom_extremities_preprocessor as pp
from dicom_extremities_preprocessor import header, pixels, rules, utils, visualize

RAW = HERE / "sample_data"
OUT = HERE / "output"

# package writes "dp"/"oblique" into ViewPosition (a CS field) -- pydicom complains once
# per written file, via warnings and via its logger
warnings.filterwarnings("ignore", message="Invalid value for VR")
logging.getLogger("pydicom").setLevel(logging.ERROR)
pd.set_option("display.width", 200)

log = logging.getLogger("demo")
log.setLevel(logging.INFO)
log.handlers = [logging.StreamHandler(sys.stdout)]
log.propagate = False

### One sample file

In [ ]:
pydicom.dcmread(RAW / "raw_03.dcm")

### Header inspection

In [ ]:
fields = ["PatientName", "StudyDate", "SeriesDescription", "StudyDescription",
          "BodyPartExamined", "Laterality", "ViewPosition",
          "PhotometricInterpretation", "Rows", "Columns"]


records = []
for p in sorted(RAW.glob("*.dcm")):
    ds = pydicom.dcmread(p, stop_before_pixels=True)
    row = {"file": p.name}
    for f in fields:
        row[f] = ds.get(f, "")
    records.append(row)

df = pd.DataFrame(records)
df


Wrong or missing in that table:

- `raw_08`: `BodyPartExamined = HAND`, but "Handgelenk" is a wrist not finger joints
- `raw_03`: `StudyDescription = Fuß` on a hand 
- `raw_02`: `ViewPosition = RL` = short for lateral 
- `raw_03`: `LLO` + no `Laterality` -> side rule says left but in image both hands
- `Laterality` often empty 

->categories get derived, tags are not reliable

## 1. Read headers

In [ ]:
# tags from rules.yaml via SimpleITK, no pixel data
# no suffix filter (raw data often has none), unreadable files got skipped
cfg = rules.load_rules()
df = header.scan_metadata(RAW, cfg["tags"], log)

df[["filename_old", "pat_id", "study_date", "series_description", "study_description",
    "body_part_examined", "laterality", "view_position", "photometric_interpretation"]]

In [ ]:
# impression of images raw
_=visualize.visualize_samples(df.sort_values("filename_old"),
                                info_cols=["filename_old", "series_description"], ncols=4)

### Umlauts

SimpleITK gives raw bytes, Python decodes them as UTF-8. Eg ("ä") becomes `U+DCE4`: matches no rule, cannot go into a CSV. 

in the cohort hits "Hand schräg"

In [ ]:
import SimpleITK as sitk

reader = sitk.ImageFileReader()
reader.SetFileName(str(RAW / "raw_05.dcm"))
reader.ReadImageInformation()
value = reader.GetMetaData("0008|103e") # SeriesDescription

print(repr(value))
print(repr(header.fix_umlauts(value)))
print(repr(rules.normalize_text(value))) # pattern "schraeg" misses
print(repr(rules.normalize_text(header.fix_umlauts(value)))) # pattern hits

## 2. Categorize in Hierarchy

Hierarchy:
- bodypart: free text > tag
- view: free text > tag, hands only
- side: `Laterality` > `ViewPosition` > free text

In [ ]:
df2 = rules.categorize(df, cfg, log)

df2[["filename_old", "series_description", "bodypart_new", "laterality_new",
     "view_position_new", "photometric_interpretation_new", "filename_new"]]

- `raw_08` "Handgelenk dp L" -> `O`, not `H` (`hand.no_joint_image`: wrist, forearm,
  single finger out of the hand category -- recategorised, not deleted)
- `raw_02` "Hand lat" -> `oblique`, not `lat` (`hand.lat_is_oblique`, checked on 800
  images; for feet `lat` is a real lateral, so the rule is hand-only)
- `raw_03` "Hand Zitherstellung" + study "Fuß" -> stays a hand (series description names
  the image, study description only the visit)

Labels stay fuzzy: `raw_02` carries a burned-in "pa", `raw_08` shows a whole hand.
Decided by the text. 
Weigh it differently -> `rules.yaml` 

## 3. Select

No recognized side -> dropped. Pairing key is (pat_id, study_date, side), and the score
table lists every joint per side.

In [ ]:
selected = {v: rules.select(df2, log, bodypart="H", view=v) for v in ("dp", "oblique")}

selected["oblique"][["filename_old", "series_description", "laterality_new",
                     "filename_new_dupl"]]

## Two hands on one image

No tag indicates directly that two hands lie on one image, and `ViewPosition` does not give it away either. 

`detect_bilateral` therefore decides on the pixels.

### `shows_two_hands`

The image is compressed into a 1D profile across its width: for every column the share
of pixels that count as tissue. One hand leaves one hump, two hands leave two with a
valley between them.

1. **Prepare** -- downscale by stride to ~400 px, stretch the contrast to [0, 1] between
   the 2nd and 98th percentile.
2. **Pixel** (`min_brightness`, 0.35) -- a pixel is tissue above
   `max(min_brightness, p60)`. The percentile lifts the threshold on brightly exposed
   images; the side effect is that max 40 % of the pixels can ever count as tissue.
3. **Column** (`min_column_fill`, 6 %) -- the mean over the rows gives the tissue share
   per column, smoothed with a moving average over width/40 columns. Above the threshold
   the column is occupied, below it is free; consecutive occupied columns form a block.
4. **Block** (`min_block_width`, 12 % of the image width) -- narrower blocks are dropped.
   A labelling marker is not a hand.
5. **Gap** (`min_gap`, 4 % of the image width) -- two surviving blocks closer than that
   are merged back into one. Spread fingers leave a valley as well, and it must not be
   read as a second hand.
6. Two or more blocks left over -> `True`, the image shows two hands.

Three levels, pixel -> column -> block, all coarse on purpose: the only question is one
block of tissue or two.

In [ ]:
# same steps as pixels.shows_two_hands, up to the smoothed profile
fig, axes = plt.subplots(2, 2, figsize=(9, 6), gridspec_kw={"height_ratios": [3, 1]})
for col, name in enumerate(["raw_02.dcm", "raw_03.dcm"]):
    px = pydicom.dcmread(RAW / name).pixel_array
    a = px[::2, ::2].astype(np.float32)
    lo, hi = np.percentile(a, [2, 98])
    a = np.clip((a - lo) / (hi - lo), 0, 1)
    share = (a > max(0.35, float(np.percentile(a, 60)))).mean(axis=0) # min_brightness
    w = max(3, len(share) // 40)
    profile = np.convolve(share, np.ones(w) / w, mode="same")

    axes[0][col].imshow(px, cmap="gray")
    axes[0][col].set_title(f"{name} -- shows_two_hands: {pixels.shows_two_hands(px)}")
    axes[0][col].axis("off")
    axes[1][col].fill_between(range(len(profile)), profile, color="0.4")
    axes[1][col].axhline(0.06, color="crimson", lw=1) # min_column_fill
    axes[1][col].set_xlim(0, len(profile))
    axes[1][col].set_ylabel("tissue share")
fig.tight_layout()

Red line = `min_column_fill`.

`raw_03` -> `laterality_new = "B"`. The writing step cuts it in the middle: the left half
is the left hand, the right half is the right hand and gets mirrored on top of that.

Reading pixels is expensive, so only candidates are checked. A candidate is landscape (`min_aspect_ratio: 1.0`, i.e. `columns/rows > 1`)
and already carries a side (`L` or `R`).

`shows_two_hands` itself knows nothing about the orientation -- it looks for two blocks
across the width, wherever they sit. The gate sits in `detect_bilateral`, and it earns
its place twice: it saves the pixel reads, and it keeps false positives out (on 400
written single hands of this cohort, all portrait, the check calls 3 of them bilateral --
spread fingers). The real limit: two hands lying *above* each other are invisible to a
column profile, and `split_dicom` cuts vertically.

## 4. Run

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)

pp.run(RAW, OUT, bodypart="H", views=("dp", "oblique"), drop_unpaired=False)

In [ ]:
sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.suffix != ".log")

7 files out of 8: wrist dropped out, bilateral one became two.

`pairs.csv`: one row = one case = one hand at one visit, not one image.

In [ ]:
pd.read_csv(OUT / "csvs" / "pairs.csv").fillna("")

### Pixels, before and after

In [ ]:
# nothing rotated, cropped or resized beyond this
# source is only read, changes happen on a copy in memory
examples = [("raw_01.dcm", "1_20200202_H_L_dp_MTwo_0.dcm", "MONOCHROME1 inverted"),
            ("raw_06.dcm", "3_20200303_H_R_dp_MTwo_0.dcm", "right hand mirrored"),
            ("raw_03.dcm", "2_20200101_H_L_oblique_MTwo_0_split.dcm", "bilateral split")]

fig, axes = plt.subplots(2, 3, figsize=(10, 8))
for col, (source, result, what) in enumerate(examples):
    view = "dp" if "_dp_" in result else "oblique"
    for row, path in enumerate([RAW / source, OUT / "dicoms" / view / result]):
        axes[row][col].imshow(pydicom.dcmread(path).pixel_array, cmap="gray")
        axes[row][col].axis("off")
    axes[0][col].set_title(f"{source}\n{what}", fontsize=9)
    axes[1][col].set_title(result, fontsize=8)
fig.tight_layout()

### Header of a written file

Same file as at the top (`raw_03.dcm`), left half, after the run:
BodyPartExamined / ViewPosition / Laterality now hold the derived categories,
ReferringPhysicianName the new name, Columns is halved.

In [ ]:
pydicom.dcmread(OUT / "dicoms" / "oblique" / "2_20200101_H_L_oblique_MTwo_0_split.dcm",
                stop_before_pixels=True)

In [ ]:
json.loads((OUT / "provenance.json").read_text())